In [4]:
# Import Required Libraries
import os

DATASETS_ROOT = "dataSets"

if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Error: Datasets folder '{DATASETS_ROOT}' not found.")
    else:
        # The project requires analyzing at least 5 datasets
        datasets = [
            d
            for d in os.listdir(DATASETS_ROOT)
            if os.path.isdir(os.path.join(DATASETS_ROOT, d))
        ]

        for dataset_name in datasets:
            path = os.path.join(DATASETS_ROOT, dataset_name)
            print(f"Processing dataset: {dataset_name}") #testing dataset name

Processing dataset: indianexpress
Processing dataset: universityherald
Processing dataset: kotiliesi
Processing dataset: ruoka_fi
Processing dataset: theguardian


In [7]:
# Import Required Libraries
import os
import re
from bs4 import BeautifulSoup
from collections import Counter

DATASETS_ROOT = "dataSets"
TARGET_TAGS = ["title", "h1", "h2", "h3", "h4", "p", "a", "strong", "b", "em"]


def get_ground_truth(soup):
    """Extracts keywords from meta tags as per newspaper dataset structure."""
    gt_keywords = set()

    # Extract from standard meta keywords and news_keywords
    meta_tags = soup.find_all("meta", attrs={"name": True})
    for meta in meta_tags:
        name = meta.get("name", "").lower()
        if name in ["keywords", "news_keywords"] and meta.get("content"):
            keywords = [k.strip().lower() for k in meta["content"].split(",")]
            gt_keywords.update(keywords)

    # Extract from OpenGraph article tags
    property_tags = soup.find_all("meta", attrs={"property": True})
    for meta in property_tags:
        prop = meta.get("property", "").lower()
        if prop == "article:tag" and meta.get("content"):
            gt_keywords.add(meta["content"].strip().lower())

    return list(gt_keywords)


def analyze_local_dataset(dataset_dir):
    tag_counts = Counter()
    total_keywords_found = 0

    # Locate all HTML files
    html_files = []
    for root, _, files in os.walk(dataset_dir):
        for file in files:
            if file.lower().endswith((".htm", ".html")):
                html_files.append(os.path.join(root, file))

    print()
    print("*" * 75)
    print(f"Analyzing {len(html_files)} files in {dataset_dir}...")

    for file_path in html_files:
        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                soup = BeautifulSoup(f.read(), "lxml")
        except Exception:
            continue

        gt_keywords = get_ground_truth(soup)
        if not gt_keywords:
            continue

        # 1. Analyze Standard HTML Tags
        for tag_name in TARGET_TAGS:
            elements = soup.find_all(tag_name)
            for el in elements:
                text = el.get_text().lower()
                for kw in gt_keywords:
                    # Use Regex for whole-word matching to avoid partial hits
                    if re.search(rf"\b{re.escape(kw)}\b", text):
                        tag_counts[tag_name] += 1
                        total_keywords_found += 1

        # 2. Analyze URL Components
        # Checks if ground-truth words appear in the file path/URL string
        path_text = file_path.lower()
        for kw in gt_keywords:
            if kw in path_text:
                tag_counts["URL path"] += 1
                total_keywords_found += 1

    return tag_counts, total_keywords_found


if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Error: Datasets folder '{DATASETS_ROOT}' not found.")
    else:
        # The project requires analyzing at least 5 datasets
        datasets = [
            d
            for d in os.listdir(DATASETS_ROOT)
            if os.path.isdir(os.path.join(DATASETS_ROOT, d))
        ]

        for dataset_name in datasets:
            path = os.path.join(DATASETS_ROOT, dataset_name)
            stats, total_found = analyze_local_dataset(path)


***************************************************************************
Analyzing 658 files in dataSets/indianexpress...

***************************************************************************
Analyzing 600 files in dataSets/universityherald...

***************************************************************************
Analyzing 420 files in dataSets/kotiliesi...

***************************************************************************
Analyzing 400 files in dataSets/ruoka_fi...

***************************************************************************
Analyzing 841 files in dataSets/theguardian...


In [8]:
import os
import re
from bs4 import BeautifulSoup
from collections import Counter

# Project Configuration
DATASETS_ROOT = "dataSets"
# Required tags per Project 1 specifications
TARGET_TAGS = ["title", "h1", "h2", "h3", "h4", "p", "a", "strong", "b", "em"]


def get_ground_truth(soup):
    """Extracts keywords from meta tags as per newspaper dataset structure."""
    gt_keywords = set()

    # Extract from standard meta keywords and news_keywords
    meta_tags = soup.find_all("meta", attrs={"name": True})
    for meta in meta_tags:
        name = meta.get("name", "").lower()
        if name in ["keywords", "news_keywords"] and meta.get("content"):
            keywords = [k.strip().lower() for k in meta["content"].split(",")]
            gt_keywords.update(keywords)

    # Extract from OpenGraph article tags
    property_tags = soup.find_all("meta", attrs={"property": True})
    for meta in property_tags:
        prop = meta.get("property", "").lower()
        if prop == "article:tag" and meta.get("content"):
            gt_keywords.add(meta["content"].strip().lower())

    return list(gt_keywords)


def analyze_local_dataset(dataset_dir):
    tag_counts = Counter()
    total_keywords_found = 0

    # Locate all HTML files
    html_files = []
    for root, _, files in os.walk(dataset_dir):
        for file in files:
            if file.lower().endswith((".htm", ".html")):
                html_files.append(os.path.join(root, file))

    print()
    print("*" * 75)
    print(f"Analyzing {len(html_files)} files in {dataset_dir}...")

    for file_path in html_files:
        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                soup = BeautifulSoup(f.read(), "lxml")
        except Exception:
            continue

        gt_keywords = get_ground_truth(soup)
        if not gt_keywords:
            continue

        # 1. Analyze Standard HTML Tags
        for tag_name in TARGET_TAGS:
            elements = soup.find_all(tag_name)
            for el in elements:
                text = el.get_text().lower()
                for kw in gt_keywords:
                    # Use Regex for whole-word matching to avoid partial hits
                    if re.search(rf"\b{re.escape(kw)}\b", text):
                        tag_counts[tag_name] += 1
                        total_keywords_found += 1

        # 2. Analyze URL Components
        # Checks if ground-truth words appear in the file path/URL string
        path_text = file_path.lower()
        for kw in gt_keywords:
            if kw in path_text:
                tag_counts["URL path"] += 1
                total_keywords_found += 1

    return tag_counts, total_keywords_found


def generate_report(counts, total, dataset_name):
    """Produces the ranked list and importance scores."""
    print(f"\n{'===== REPORT: ' + dataset_name + ' =====':^75}")
    print(
        f"{'HTML Tag':<15} | {'Occurrences':<12} | {'Percentage':<12} | {'Importance Score':<12}"
    )
    print("-" * 75)

    # Ranking top 10 tags by frequency
    sorted_tags = counts.most_common(10)
    if not sorted_tags:
        print("No keyword occurrences found in the analyzed tags.")
        return

    # Normalization: highest occurrence gets 1.0
    max_occ = sorted_tags[0][1]

    for tag, occ in sorted_tags:
        percentage = (occ / total) * 100 if total > 0 else 0
        importance_score = round(occ / max_occ, 2)

        tag_display = f"<{tag}>" if tag != "URL path" else tag
        print(
            f"{tag_display:<15} | {occ:<12} | {percentage:>10.2f}% | {importance_score:>15.2f}"
        )


if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Error: Datasets folder '{DATASETS_ROOT}' not found.")
    else:
        # The project requires analyzing at least 5 datasets
        datasets = [
            d
            for d in os.listdir(DATASETS_ROOT)
            if os.path.isdir(os.path.join(DATASETS_ROOT, d))
        ]

        for dataset_name in datasets:
            path = os.path.join(DATASETS_ROOT, dataset_name)
            stats, total_found = analyze_local_dataset(path)
            generate_report(stats, total_found, dataset_name)


***************************************************************************
Analyzing 658 files in dataSets/indianexpress...

                     ===== REPORT: indianexpress =====                     
HTML Tag        | Occurrences  | Percentage   | Importance Score
---------------------------------------------------------------------------
<a>             | 31679        |      71.89% |            1.00
<p>             | 6850         |      15.55% |            0.22
<h4>            | 1183         |       2.68% |            0.04
<title>         | 1178         |       2.67% |            0.04
<h1>            | 972          |       2.21% |            0.03
<h2>            | 904          |       2.05% |            0.03
URL path        | 630          |       1.43% |            0.02
<strong>        | 579          |       1.31% |            0.02
<em>            | 52           |       0.12% |            0.00
<b>             | 34           |       0.08% |            0.00

*************************